# 01. Подготовка данных

Приведение исходной выгрузки к виду, пригодному для анализа: стандартизация наименований курсов, восстановление пропущенных значений возраста, укрупнение категориальных признаков.

Источник данных — выгрузка Kodland, международной онлайн-школы программирования для детей. Данные предоставлены компанией и используются с её разрешения: значения изменены, идентификаторы синтетические, персональные данные в выборке отсутствуют. Единица наблюдения — ученик, впервые начавший обучение.

Логика преобразований сохранена в исходном виде. Код разбит на логические ячейки и снабжён пояснениями, однако сами процедуры не изменялись; замечания к ним вынесены в `docs/preprocessing_notes.md`. Все последующие операции — фильтрация выборки, обработка выбросов, нормировка рейтинга — выполнены в ноутбуке 02, что обеспечивает разделение авторского кода и внесённых доработок.

Единственное внесённое изменение — путь к файлу данных: выгрузка размещена в каталоге `data/raw/`.

In [1]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

## Загрузка данных

В качестве индекса используется идентификатор ученика.

In [2]:
df = pd.read_excel('../data/raw/students.xlsx', index_col=0)
df.head()

,gender,utm_source,age,payment_course,first_course_name,course_format,first_payment_type,group_id,first_course_start_date,first_course_end_date,group_churn_date,missed_classes,incomplete_hw,rating,cs_tickets,second_course_payment_date
id,,,,,,,,,,,,,,,,
36056830,UNKNOWN,flocktory,13.0,Unity,[858]3D игры на Unity[None][13-17][90 min][32 ...,Индивидуальная группа,Full,28733,2024-01-20,2024-10-05,2024-08-24,0.038462,0.925926,1,11,2024-03-17
36262276,WOMAN,ig,12.0,Цифровой дизайн,"[610] Дизайн цифровых миров [2022][10-12][50m,...",Микро-группа,Full,30350,2024-03-28,2024-10-31,2024-07-18,0.058824,0.909091,3,18,2024-11-09
36288588,UNKNOWN,ig,13.0,Python,[721] Python Base Internship LVL 1 [2022][13+]...,Стандартная группа,Full,30591,2024-04-07,2024-11-10,2024-08-04,0.000000,0.666667,2,14,NaT
36812572,MAN,website,15.0,[887] Python LVL 2 [2023][13+][90min][32L][Ru]...,[887]Python Pro[None][13-17][90 min][32 L][CIS...,Premium Group,Internal installment,35068,2024-08-11,2025-03-23,NaT,0.093750,0.406250,3,28,NaT
36355456,UNKNOWN,flocktory,7.0,FunTech Explorers,[1053]FunTech Explorers[2023][5-7][RU],Мини-группа | Короткая,Bank installment,30904,2024-04-15,2024-11-23,2024-06-22,0.100000,0.000000,1,12,NaT


## Состав выгрузки

Выгрузка содержит шестнадцать полей: демографические характеристики, наименование курса и формат обучения, тип первой оплаты, даты начала и завершения обучения, поведенческие показатели за период курса и дату приобретения второго курса.

Пропуски сосредоточены в четырёх полях: канал привлечения, возраст, дата выбытия из группы и дата повторной покупки. В двух последних случаях пропуск не является ошибкой и означает отсутствие события: если ученик не выбывал из группы или не приобретал второй курс, соответствующая дата не заполняется.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7295 entries, 36056830 to 36377084
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   gender                      7295 non-null   object        
 1   utm_source                  6537 non-null   object        
 2   age                         6993 non-null   float64       
 3   payment_course              7221 non-null   object        
 4   first_course_name           7295 non-null   object        
 5   course_format               7295 non-null   object        
 6   first_payment_type          7295 non-null   object        
 7   group_id                    7295 non-null   int64         
 8   first_course_start_date     7295 non-null   datetime64[ns]
 9   first_course_end_date       7295 non-null   datetime64[ns]
 10  group_churn_date            3949 non-null   datetime64[ns]
 11  missed_classes              7295 non-null   float6

In [4]:
df.isna().sum()

gender                           0
utm_source                     758
age                            302
payment_course                  74
first_course_name                0
course_format                    0
first_payment_type               0
group_id                         0
first_course_start_date          0
first_course_end_date            0
group_churn_date              3346
missed_classes                   0
incomplete_hw                    0
rating                           0
cs_tickets                       0
second_course_payment_date    5031
dtype: int64

In [5]:
df['first_course_name'].value_counts().head(50)

first_course_name
[721] Python Base Internship LVL 1 [2022][13+],90m][32L][Ru][Actual]            1371
[711] Волшебство программирования в Scratch [2022][8-10]in][32L][Ru][Actual]    1049
[983] Roblox Game Developer [2023][10-12][50, 90m][32L][Ru][Actual]                  956
[1233]Python LVL1[None][12-17][90 min][32 L][CIS][actual]                            745
[949] Roblox Game Developer [2023][8-9][60m][32L][Ru][Actual]                        691
[1056]Цифровое творчество. Уровень 2[None][10-12][50 min][32 L][CIS][actual]         466
[611]Дизайн цифровых миров PRO[None][13-17][90 min][32 L][CIS][not assign]           444
[610] Дизайн цифровых миров [2022][10-12],60m][32L][Ru][Actual]                  310
[1053]FunTech Explorers[2023][5-7][RU]                                               260
[858]3D игры на Unity[None][13-17][90 min][32 L][CIS][actual]                        188
[1264] Roblox Game Developer LVL 1 [2024][10-12][50, 90m][32L][Ru][Actual]           161
[609] Web Desig

## Стандартизация наименований курсов

Наименования поступают из внутренней системы вместе с техническими метками: идентификатором, годом запуска, возрастной группой, длительностью занятия, количеством уроков, регионом и статусом программы. Для анализа значимо только наименование программы, вследствие чего содержимое квадратных скобок удаляется.

После очистки остаётся 24 варианта, часть из которых соответствует одной и той же программе под различными наименованиями: русско- и англоязычная версии Scratch, испаноязычная версия Roblox, два наименования Unity. Словарь соответствий сводит их к 19 категориям.

In [6]:
import re

def clean_course_name(x):
    if pd.isna(x):
        return x
    
    # удаляем все, что в квадратных скобках
    x = re.sub(r'\[.*?\]', '', x)
    x = x.strip()
    x = x.lower()
    
    return x

df['course_clean'] = df['first_course_name'].apply(clean_course_name)
print('Уникальных курсов:', df['course_clean'].nunique())

course_counts = df['course_clean'].value_counts()
print(course_counts.head(50))

Уникальных курсов: 24
course_clean
roblox game developer                    1648
python base internship lvl 1             1371
волшебство программирования в scratch    1049
python lvl1                               746
цифровое творчество. уровень 2            466
дизайн цифровых миров pro                 444
дизайн цифровых миров                     310
roblox game developer lvl 1               289
funtech explorers                         260
3d игры на unity                          188
web design                                141
python pro                                103
компьютерная грамотность                   62
графический дизайн                         38
early math level 1                         38
roblox game developer lvl 2                35
early math level 2                         33
early math level 3                         19
3d игры на unity pro                       17
программирование на javascript             17
python lvl3                                17

In [7]:
# создаем словарь
course_map = {
    'python base internship lvl 1'          :'python1',
    'python lvl1'                           :'python1',
    'python pro'                            :'pythonPRO',
    'python lvl3'                           :'python3',
    'волшебство программирования в scratch' :'scratch',
    'magic of code with scratch'            :'scratch',
    'roblox game developer'                 :'roblox_base',
    'roblox game developer lvl 1'           :'roblox1',
    'roblox game developer lvl 2'           :'roblox2',
    'desarrollador de juegos roblox'        :'roblox_base',
    'дизайн цифровых миров'                 :'animation',
    'web design'                            :'web_design',
    'графический дизайн'                    :'graphic_design',
    'дизайн цифровых миров pro'             :'designPRO',
    'funtech explorers'                     :'funtech',
    '3d игры на unity'                      :'unity',
    'unity game developer'                  :'unity',
    '3d игры на unity pro'                  :'unityPRO',
    'программирование на javascript'        :'javascript',
    'early math level 1'                    :'math1',
    'early math level 2'                    :'math2',
    'early math level 3'                    :'math3',
    'компьютерная грамотность'              :'computer_literacy',
    'цифровое творчество. уровень 2'        :'animation'
}

df['course_group'] = df['course_clean'].map(course_map)
assert df['course_group'].isna().sum() == 0, "есть незамапленные курсы"
course_counts = df['course_group'].value_counts()
print('Уникальных курсов:', df['course_group'].nunique())
df['course_group'] = df['course_group'].astype('category')

print(course_counts.head(50))

Уникальных курсов: 19
course_group
python1              2117
roblox_base          1649
scratch              1050
animation             776
designPRO             444
roblox1               289
funtech               260
unity                 190
web_design            141
pythonPRO             103
computer_literacy      62
graphic_design         38
math1                  38
roblox2                35
math2                  33
math3                  19
unityPRO               17
javascript             17
python3                17
Name: count, dtype: int64


## Восстановление возраста

Возраст известен не для всех наблюдений, однако допускает восстановление: наименование курса содержит целевую возрастную группу. Для диапазона вида `[10-12]` используется середина интервала, для обозначения `[13+]` — середина между нижней границей и семнадцатью годами.

Значения вне диапазона 5–17 лет квалифицируются как ошибки ввода и обнуляются, оставшиеся пропуски заполняются медианным значением по курсу.

In [8]:
def get_age(x):
    if pd.isna(x):
        return np.nan

    # [10-12]
    r = re.search(r'\[(\d{1,2})-(\d{1,2})\]', x)
    if r:
        a, b = map(int, r.groups())
        return (a + b) / 2

    # [13+]
    r = re.search(r'\[(\d{1,2})\+\]', x)
    if r:
        a = int(r.group(1))
        return (a + 17) / 2

    return np.nan

# 1. достаём возраст из курса
df['age'] = df['age'].fillna(df['first_course_name'].apply(get_age))

# 2. чистим выбросы (оставляем только 5–17)
df.loc[(df['age'] < 5) | (df['age'] > 17), 'age'] = np.nan

# 3. заполняем по курсу
df['age'] = df.groupby('course_group')['age'].transform(lambda x: x.fillna(x.median()))

df['age'] = df['age'].round().astype(int)
df['age'].value_counts().head(50)

age
11    1062
10    1039
12     937
9      930
8      830
13     625
14     595
7      526
15     407
16     145
6      117
17      47
5       35
Name: count, dtype: int64

## Укрупнение каналов привлечения

Поле содержит множественные варианты написания одного и того же канала: значения `ig`, `instagram`, `instagram_page`, `instragram` относятся к единому источнику. Присутствуют также технические артефакты — незаполненный шаблон `{{site_source_name}}` и его URL-кодированная форма.

Словарь соответствий сводит 39 исходных значений к 11 каналам. Пропущенные значения относятся к категории `other`.

In [9]:
df['utm_source'].value_counts()

utm_source
ig                              1811
flocktory                       1566
fb                               875
yandex                           772
vk                               328
WA                               211
website                          176
vk_leadform                      171
instagram_page                   103
Flocktory2.0                      97
social                            92
socialmediamod                    72
influencer                        63
telegram_page                     30
tg                                26
{{site_source_name}}              23
instagram                         21
vk_page                           17
facebook                          14
flocktoryKZ                       12
an                                10
tg-ads                             7
blog                               5
yandex.zen                         5
instragram                         5
vk_mp                              4
%7B%7Bsite_source_name%7D%7

In [10]:
utm_map = {
    'ig'                           :'meta',
    'instagram'                    :'meta',
    'instagram_page'               :'meta',
    'instragram'                   :'meta',
    'fb'                           :'meta',
    'facebook'                     :'meta',
    'facebook_page'                :'meta',
    'fbb'                          :'meta',

    'vk'                           :'vk',
    'vk_page'                      :'vk',
    'vk_leadform'                  :'vk',
    'vk_mp'                        :'vk',
    'vkgen_voronka'                :'vk',

    'yandex'                       :'yandex',
    'yandex.zen'                   :'yandex',
    'yandex_dz'                    :'yandex',
    'dzen'                         :'yandex',

    'tg'                           :'telegram',
    'tg-ads'                       :'telegram',
    'telegram_page'                :'telegram',

    'WA'                           :'whatsapp',

    'flocktory'                    :'referral',
    'Flocktory2.0'                 :'referral',
    'flocktoryKZ'                  :'referral',
    'affise'                       :'referral',

    'influencer'                   :'influencer',

    'website'                      :'website',
    '{{site_source_name}}'         :'website',
    '%7B%7Bsite_source_name%7D%7D' :'website',

    'blog'                         :'content',
    'youtube'                      :'content',
    'tiktok'                       :'content',

    'social'                       :'social',
    'socialmediamod'               :'social',

    'an'                           :'other',
    'study_ru'                     :'other',
    'sng'                          :'other',
    'quiz'                         :'other',
    'google'                       :'other'
}
df['utm_source_group'] = df['utm_source'].map(utm_map).fillna('other')
df['utm_source_group'] = df['utm_source_group'].astype('category')
df['utm_source_group'].value_counts()

utm_source_group
meta          2832
referral      1676
yandex         779
other          772
vk             523
whatsapp       211
website        203
social         164
influencer      63
telegram        63
content          9
Name: count, dtype: int64

## Укрупнение формата обучения и типа оплаты

Форматы группируются по численности и ценовому уровню: стандартные и мини-группы отнесены к категории `standard`, микро-группы и премиальные — к `premium`, индивидуальные занятия — к `individual`.

Типы оплаты сведены к трём категориям: полная оплата, банковская рассрочка и внутренняя рассрочка. Значение `Downpayment` в словарь соответствий не включено, вследствие чего для соответствующих наблюдений категория остаётся незаполненной; решение по ним принимается в ноутбуке 02.

In [11]:
df['course_format'].value_counts().head(50)

format_map = {
    'Стандартная группа'     :'standard',
    'Мини-группа'            :'standard',
    'Мини-группа | Средняя'  :'standard',
    'Мини-группа | Короткая' :'standard',
    'Микро-группа'           :'premium',
    'Premium Group'          :'premium',
    'Small group'            :'premium',
    'Large Premium Group'    :'premium',
    'Индивидуальная группа'  :'individual',
    'Индивидуальная KLP'     :'individual'
}

In [12]:
df['course_format_group'] = df['course_format'].map(format_map)
df['course_format_group'].astype('category')

df['course_format_group'].value_counts()

course_format_group
premium       3579
standard      2996
individual     720
Name: count, dtype: int64

In [13]:
df['first_payment_type'].value_counts().head(50)

payment_map = {
    'Full'                  :'full',
    'Bank installment'      :'bank_installment',
    'Internal installment'  :'internal_installment',
    'Undefined'             :'internal_installment',
    'Subscription (auto)'   :'internal_installment'
}

In [14]:
df['payment_type_group'] = df['first_payment_type'].map(payment_map)
df['payment_type_group'].astype('category')

df['payment_type_group'].value_counts().head(50)

payment_type_group
full                    3217
bank_installment        2415
internal_installment    1562
Name: count, dtype: int64

## Результат подготовки

К шестнадцати исходным полям добавлены пять производных: очищенное и сгруппированное наименование курса, укрупнённые канал привлечения, формат обучения и тип оплаты.

Далее формируется рабочий срез `data`, включающий отобранные для анализа поля с приведением категориальных признаков к соответствующему типу.

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7295 entries, 36056830 to 36377084
Data columns (total 21 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   gender                      7295 non-null   object        
 1   utm_source                  6537 non-null   object        
 2   age                         7295 non-null   int64         
 3   payment_course              7221 non-null   object        
 4   first_course_name           7295 non-null   object        
 5   course_format               7295 non-null   object        
 6   first_payment_type          7295 non-null   object        
 7   group_id                    7295 non-null   int64         
 8   first_course_start_date     7295 non-null   datetime64[ns]
 9   first_course_end_date       7295 non-null   datetime64[ns]
 10  group_churn_date            3949 non-null   datetime64[ns]
 11  missed_classes              7295 non-null   float6

In [16]:
data = df[[
    'gender',
    'age',
    'course_group',
    'utm_source_group',
    'course_format_group',
    'payment_type_group',
    'first_course_start_date',
    'first_course_end_date',
    'group_churn_date',
    'missed_classes',
    'incomplete_hw',
    'rating',
    'cs_tickets',
    'second_course_payment_date',
    'first_payment_type'
]].copy()

cat_cols = [
    'gender',
    'course_group',
    'utm_source_group',
    'course_format_group',
    'payment_type_group'
]

for col in cat_cols:
    data[col] = data[col].astype('category')

In [17]:
data['rating']

id
36056830     1
36262276     3
36288588     2
36812572     3
36355456     1
            ..
36049312    11
36291816     5
36937826     7
37224932     7
36377084    16
Name: rating, Length: 7295, dtype: int64

## Сохранение результата

Результат сохраняется в базу SQLite. Записывается полный датафрейм, а не срез `data`: для последующего восстановления размера учебной группы требуется поле `group_id`, в срез не включённое.

In [18]:
import sqlite3
from contextlib import closing
from pathlib import Path

DB_PATH = Path('../data/churn.db')
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

with closing(sqlite3.connect(DB_PATH)) as conn:
    df.to_sql('students_prepared', conn, if_exists='replace', index=True, index_label='id')
    conn.commit()
    saved = pd.read_sql_query('SELECT COUNT(*) AS rows FROM students_prepared', conn)

print(f'students_prepared: {len(df):,} строк, {df.shape[1]} колонок')
saved

students_prepared: 7,295 строк, 21 колонок


,rows
0,7295
